## Example 1: Basic DataFrame Schema Validation

In [ ]:
import pandera as pa
import pandas as pd

# Example 1: Basic schema validation
print("Example 1 - Basic validation")

schema = pa.DataFrameSchema({
    "name": pa.Column(pa.String, checks=pa.Check.str_length(min_value=2)),
    "age": pa.Column(pa.Int, checks=pa.Check.between(0, 120)),
    "score": pa.Column(pa.Float, checks=pa.Check.in_range(0.0, 100.0))
})

df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "age": [25, 30, 35],
    "score": [85.5, 92.0, 78.5]
})

validated = schema.validate(df)
print(f"Validation successful: {validated.shape}")
assert validated.shape == (3, 3)
print(f"Types: {validated.dtypes.to_dict()}")

print("Example 1 completed")

## Example 2: Custom Validation Checks

In [ ]:
import pandera as pa
import pandas as pd

# Example 2: Custom checks
print("Example 2 - Custom checks")

schema = pa.DataFrameSchema({
    "email": pa.Column(pa.String, checks=pa.Check(lambda s: s.str.contains('@'))),
    "price": pa.Column(pa.Float, checks=[pa.Check.gt(0), pa.Check.lt(1000)])
})

df = pd.DataFrame({
    "email": ["alice@test.com", "bob@example.org"],
    "price": [19.99, 29.99]
})

validated = schema.validate(df)
print(f"Custom validation: {validated.shape}")

# Lambda check
custom_col = pa.Column(pa.Float, checks=pa.Check(lambda s: s.mean() > 10))
print(f"Lambda check configured")

print("Example 2 completed")

## Example 3: DataFrameModel Class Schema

In [ ]:
import pandera as pa
import pandas as pd
from pandera.typing import Series

# Example 3: Class-based schema
print("Example 3 - DataFrameModel")

class UserSchema(pa.DataFrameModel):
    user_id: Series[int] = pa.Field(gt=0, unique=True)
    username: Series[str] = pa.Field(str_length={'min_value': 3})
    age: Series[int] = pa.Field(ge=13, le=120)

df = pd.DataFrame({
    "user_id": [1, 2, 3],
    "username": ["alice", "bob", "charlie"],
    "age": [25, 30, 22]
})

validated = UserSchema.validate(df)
print(f"Class validation: {validated.shape}")
print(f"Schema: {UserSchema.__name__}")

print("Example 3 completed")

## Example 4: Error Handling and Reporting

In [ ]:
import pandera as pa
import pandas as pd

# Example 4: Error handling
print("Example 4 - Error handling")

schema = pa.DataFrameSchema({
    "price": pa.Column(pa.Float, checks=pa.Check.gt(0)),
    "quantity": pa.Column(pa.Int, checks=pa.Check.in_range(1, 100))
})

invalid_df = pd.DataFrame({"price": [10.0, -5.0], "quantity": [5, 200]})

try:
    schema.validate(invalid_df, lazy=True)
except pa.errors.SchemaErrors as e:
    print(f"Errors found: {len(e.failure_cases)}")
    for err in e.failure_cases.itertuples():
        print(f"  {err.column}: {err.check}")
    assert set(e.failure_cases["column"]) == {"price", "quantity"}
else:
    raise AssertionError("invalid_df passed validation")

valid_df = pd.DataFrame({"price": [10.0, 20.0], "quantity": [5, 10]})
validated = schema.validate(valid_df)
print(f"\nValid data: {validated.shape}")

print("Example 4 completed")

## Example 5: Type Coercion and Index Validation

In [ ]:
import pandera as pa
import pandas as pd

# Example 5: Coercion and index validation
print("Example 5 - Coercion and index")

# Type coercion
coerce_schema = pa.DataFrameSchema({
    "numbers": pa.Column(pa.Float, coerce=True),
    "dates": pa.Column(pa.DateTime, coerce=True)
})

raw_df = pd.DataFrame({"numbers": ["1.5", "2.0"], "dates": ["2023-01-01", "2023-01-02"]})
coerced = coerce_schema.validate(raw_df)
print(f"Coerced types: {coerced.dtypes.to_dict()}")

# Index validation
index_schema = pa.DataFrameSchema(
    {"value": pa.Column(pa.Float)},
    index=pa.Index(pa.String, checks=pa.Check.str_matches(r'^ID_\d+$'))
)

df = pd.DataFrame({"value": [1.0, 2.0]}, index=["ID_001", "ID_002"])
validated = index_schema.validate(df)
print(f"Index validated: {validated.shape}")

print("Example 5 completed")